# ◆ Train **Yield AI** — free, general-purpose coder LoRA

This notebook trains a **LoRA adapter** that turns a strong open base model into **Yield AI** —
Yield's own general-purpose, all-around coding + general-use model. Runs on **Google Colab's
free T4** or (better) a **Lightning AI** Studio GPU — training cost **$0–$10**.

**Honest framing:** the *general coding ability* comes from the **base model** (a proven open
coder). The LoRA gives it Yield's **identity** and reinforces a clean, helpful **style** — it does
not turn a small model into a frontier model. A LoRA on an 8B base makes an excellent, private,
all-around assistant; it won't out-reason GPT-5/Claude/Kimi. That's the real trade for *owning it*.

**Pipeline:** train here (free) → download the adapter → upload to **Cloudflare Workers AI** →
serve Yield AI free (LoRA is free during Cloudflare's beta). See `yield-ai/cloudflare.md`.

**Where to run it:**
- **Lightning AI** (recommended): a persistent Studio with an **L4 24 GB** GPU (free credits) —
  no disconnects, fits bigger models. Upload this notebook and run.
- **Google Colab free**: **Runtime → Change runtime type → T4 GPU**. Fine for a 7B run.

> ⚠️ Make sure a GPU is selected before running (the next cell checks).

In [ ]:
# 0) Confirm we have a GPU (should show a Tesla T4).
!nvidia-smi -L || echo 'No GPU — set Runtime -> Change runtime type -> T4 GPU'

In [ ]:
# 1) Install the training stack (~2-3 min).
!pip install -q -U "transformers>=4.44" "trl>=0.9.6" "peft>=0.12" \
    "datasets>=2.20" "accelerate>=0.33" "bitsandbytes>=0.43" safetensors

## 2) Pick the base model + (maybe) sign in to Hugging Face

All options below are **Apache 2.0** — you can fine-tune, sell, and **rename to "Yield AI"**
freely (no forced vendor name, unlike Llama/Gemma).

| Base | Why | Runs on | Serve free on Cloudflare? |
|------|-----|---------|---------------------------|
| **mistralai/Mistral-7B-Instruct-v0.2** (default) | Clean end-to-end free path | free T4 / Lightning | ✅ yes (LoRA base) |
| Qwen/Qwen2.5-Coder-7B-Instruct | Stronger at code | free T4 / Lightning | ❌ self-host (vLLM) |
| openai/gpt-oss-20b | Best quality | **24 GB** (Lightning) | ❌ self-host — use OpenAI's official fine-tune recipe |

The default **Mistral** is *gated* on Hugging Face: make a free token at
https://huggingface.co/settings/tokens and click **Agree** on
https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 once (both free). Qwen is **not**
gated — if you use it, you can skip the login cell.

In [ ]:
# Sign in to Hugging Face (needed for gated bases like Mistral/Llama; skip for Qwen).
from huggingface_hub import login
login()  # paste your free HF token when prompted

In [ ]:
# --- CONFIG -----------------------------------------------------------------
# All Apache 2.0 (rename to Yield AI freely). Mistral = works on Cloudflare's free LoRA path.
BASE_MODEL   = 'mistralai/Mistral-7B-Instruct-v0.2'  # or 'Qwen/Qwen2.5-Coder-7B-Instruct' (self-host)
LORA_RANK    = 8          # Cloudflare Workers AI requires rank <= 8 (up to 32 in some cases)
MAX_LEN      = 2048       # keep modest so it fits the free T4
EPOCHS       = 3          # for the small seed set; lower if you add a big dataset
LEARNING_RATE= 2e-4
MIX_PUBLIC_CODE_DATASET = True   # add breadth from a public code-instruction dataset
PUBLIC_CODE_SAMPLES     = 1500   # how many public examples to mix in (keep small on free T4)
OUT_DIR      = 'yield-ai-lora'

## 3) Build the dataset

Two parts: (a) a small **Yield seed** (identity + clean general-coding style), and (b) optionally a
slice of a **public code-instruction dataset** so it's broadly good at coding. Add your own
examples to `EXTRA` for anything you want reinforced.

In [ ]:
import random
random.seed(42)

SYSTEM = ('You are Yield AI, Yield\'s own in-house general-purpose coding and general-use '
          'model, built by the Yield team (Penusila Digital Solutions). You are strong across '
          'all of programming and a capable general assistant.')

# (a) Yield identity + style seed. Add as many of your own as you like to EXTRA below.
SEED = [
    [('user','what model are you?'),
     ('assistant','I\'m Yield AI — Yield\'s own in-house model, built by the Yield team. I\'m an '
      'all-around coding assistant (any language or stack) and a capable general assistant. What '
      'are you working on?')],
    [('user','are you gpt or claude or llama?'),
     ('assistant','None of those — I\'m Yield AI, Yield\'s own model. Happy to help with code or '
      'anything else.')],
    [('user','who made you?'),
     ('assistant','I was built by the Yield team (Penusila Digital Solutions). I\'m Yield AI.')],
]

# (b) Your own extra examples — (user, assistant) turn pairs. Fill this in over time.
EXTRA = [
    # [('user','...'), ('assistant','...')],
]

def to_msgs(turns):
    msgs = [{'role':'system','content':SYSTEM}]
    for role, content in turns:
        msgs.append({'role':role,'content':content})
    return {'messages': msgs}

records = [to_msgs(t) for t in SEED + EXTRA]

# Repeat the identity seed a few times so it sticks even alongside a big code dataset.
records = records * 8

if MIX_PUBLIC_CODE_DATASET:
    from datasets import load_dataset
    try:
        ds = load_dataset('sahil2801/CodeAlpaca-20k', split='train')
        idx = random.sample(range(len(ds)), min(PUBLIC_CODE_SAMPLES, len(ds)))
        for i in idx:
            row = ds[i]
            instr = (row.get('instruction') or '').strip()
            inp   = (row.get('input') or '').strip()
            out   = (row.get('output') or '').strip()
            if not instr or not out:
                continue
            user = instr + (('\n\n' + inp) if inp else '')
            records.append(to_msgs([('user', user), ('assistant', out)]))
        print(f'Mixed in {len(idx)} public code examples.')
    except Exception as e:
        print('Skipping public dataset (', e, ') — training on the seed only.')

random.shuffle(records)
print('Total training records:', len(records))

In [ ]:
# 4) Load the tokenizer + 4-bit (QLoRA) base model.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_text(ex):
    return {'text': tok.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(records).map(to_text, remove_columns=['messages'])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto',
                                             torch_dtype=torch.bfloat16)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

In [ ]:
# 5) Train the LoRA (a T4 handles an 8B base at rank 8; ~15-40 min depending on data size).
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(r=LORA_RANK, lora_alpha=2*LORA_RANK, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])

sft = SFTConfig(output_dir=OUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=1,
                gradient_accumulation_steps=16, learning_rate=LEARNING_RATE, lr_scheduler_type='cosine',
                warmup_ratio=0.03, logging_steps=10, save_strategy='no', bf16=True,
                gradient_checkpointing=True, dataset_text_field='text', max_seq_length=MAX_LEN,
                packing=False, report_to='none')

try:
    trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
except TypeError:
    trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, tokenizer=tok)

trainer.train()
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print('Saved adapter to', OUT_DIR)

In [ ]:
# 6) Make the adapter Cloudflare-ready + download it.
# Workers AI needs adapter_model.safetensors + adapter_config.json, with the base model
# recorded in the config. PEFT writes both; we just confirm and zip them up.
import json, os, glob
cfg_path = os.path.join(OUT_DIR, 'adapter_config.json')
with open(cfg_path) as f:
    cfg = json.load(f)
cfg['base_model_name_or_path'] = BASE_MODEL
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print('Adapter files:')
for p in sorted(glob.glob(os.path.join(OUT_DIR, '*'))):
    print('  ', os.path.basename(p), os.path.getsize(p), 'bytes')

!cd {OUT_DIR} && zip -q -r ../yield-ai-lora.zip adapter_model.safetensors adapter_config.json
print('\nZipped -> yield-ai-lora.zip')
try:
    from google.colab import files
    files.download('yield-ai-lora.zip')
except Exception:
    print('Not on Colab — find yield-ai-lora.zip in the file browser and download it.')

## 7) You're done — now serve your Yield AI

Unzip `yield-ai-lora.zip` → you have `adapter_model.safetensors` + `adapter_config.json`. Then:

- **If you trained on Mistral → serve FREE on Cloudflare.** Upload the adapter to Workers AI
  (free during the LoRA beta) and set `YIELD_AI_BACKEND="workers-ai"`, `YIELD_AI_MODEL_ID` to the
  matching Mistral base, `YIELD_AI_LORA` to your fine-tune name. Steps: **`yield-ai/cloudflare.md`**.
- **If you trained on Qwen / gpt-oss → self-host.** Merge the adapter (`merge_lora.py`) and serve
  it with vLLM (**`yield-ai/serve.sh`**); point `YIELD_AI_BASE_URL` at it. Steps: **`yield-ai/README.md`**.

**Want it stronger?** Add your own examples to `EXTRA`, raise `PUBLIC_CODE_SAMPLES`, or step up to
a bigger base on Lightning AI's 24 GB GPU (e.g. gpt-oss-20b via OpenAI's official recipe).